<a href="https://colab.research.google.com/github/Shashini294/Statistical-Learning-e22294/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Conceptual Overview & Problem Formulation

In **Structural Health Monitoring (SHM)**, sensors (e.g., strain gauges, accelerometers, or ultrasonic sensors) measure structural responses over time to estimate a continuous, bounded physical parameter representing damage severity or degradation:


$$\Theta = \theta \in [\theta_{\min}, \theta_{\max}]$$

Because non-linear degradation models often lack conjugate priors, **Bounded Grid Integration** discretizes the bounded domain $[\theta_{\min}, \theta_{\max}]$ to maintain and sequentially update the full posterior probability density function $f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)})$.

---

## Mathematical Formulation

### 1. Measurement Model & Single Likelihood Contribution

Let $Y_k$ denote the physical response measured by a sensor at time step $k$. Given latent damage parameter $\theta$, the observation is modeled as:


$$Y_k = g(\theta) + \varepsilon_k, \quad \varepsilon_k \sim \mathcal{N}(0, \sigma^2)$$

where $g(\theta)$ is the physical response function (e.g., stiffness degradation or frequency response) and $\sigma^2$ is the measurement noise variance.

The likelihood contribution of a single observation $y_k$ given $\Theta = \theta$ is:


$$L(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(y_k - g(\theta))^2}{2\sigma^2} \right)$$

### 2. Joint Likelihood for Running History

Assuming sensor noise $\varepsilon_k$ is conditionally independent given $\Theta = \theta$, the joint likelihood for the running observation vector $y^{(k)} = (y_1, y_2, \dots, y_k)$ is:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = (2\pi\sigma^2)^{-k/2} \exp\left( -\sum_{i=1}^k \frac{(y_i - g(\theta))^2}{2\sigma^2} \right)$$

### 3. Recursive Sequential Posterior Update

Starting from a bounded prior $f_{\Theta}^{(0)}(\theta)$ defined over $[\theta_{\min}, \theta_{\max}]$ (such as a uniform prior $\mathcal{U}(\theta_{\min}, \theta_{\max})$), the posterior at step $k$ is computed recursively from step $k-1$:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \cdot f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}$$

---

## Bounded Grid Discretization Algorithm

To evaluate non-analytical posterior updates computationally:

1. **Grid Discretization:** Discretize the domain $[\theta_{\min}, \theta_{\max}]$ into $M$ uniformly spaced grid points:

$$\Theta_{\text{grid}} = \{\theta_1, \theta_2, \dots, \theta_M\}$$


2. **Prior Initialization:** Initialize grid density values:

$$f^{(0)}(\theta_m) = \frac{1}{\theta_{\max} - \theta_{\min}} \quad \forall m \in \{1, \dots, M\}$$


3. **Sequential Update Loop:** At each time step $k$ upon receiving observation $y_k$:
* Compute grid likelihood vector: $L_m = L(y_k \mid \theta_m)$
* Compute unnormalized posterior: $w_m^{(k)} = L_m \cdot f^{(k-1)}(\theta_m)$
* Compute normalizing constant via trapezoidal integration:

$$I^{(k)} = \int_{\theta_{\min}}^{\theta_{\max}} w^{(k)}(\theta) \, d\theta \approx \text{trapz}(w^{(k)}, \Theta_{\text{grid}})$$


* Normalize probability density:

$$f^{(k)}(\theta_m) = \frac{w_m^{(k)}}{I^{(k)}}$$





---

## Point Estimators

At any step $k$, point estimates are extracted directly from the normalized grid array:

1. **Running Posterior Mean (Bayes Estimate under L2 Loss):**

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid y^{(k)}] \approx \int_{\theta_{\min}}^{\theta_{\max}} \theta f^{(k)}(\theta) \, d\theta \approx \text{trapz}(\Theta_{\text{grid}} \cdot f^{(k)}, \Theta_{\text{grid}})$$


2. **Running Maximum A Posteriori (MAP) Estimate:**

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta_m \in \Theta_{\text{grid}}} f^{(k)}(\theta_m)$$



---

## Python Implementation & Plotly Simulation

The following complete Python script simulates $n = 30$ sequential sensor readings, tracks the true damage parameter $\theta_{\text{true}} = 0.65$ across a bounded grid $\theta \in [0, 1]$, and plots the convergence trajectory using Plotly:

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# SIMULATION SETUP: STRUCTURAL HEALTH MONITORING (SHM)
# =====================================================================
np.random.seed(42)

# True physical parameters
theta_true = 0.65       # True damage parameter (e.g. 65% loss of stiffness)
noise_sigma = 0.08      # Sensor noise standard deviation
n_steps = 30            # Number of sequential sensor measurements

# Define bounded grid
theta_min, theta_max = 0.0, 1.0
grid_points = 500
theta_grid = np.linspace(theta_min, theta_max, grid_points)

# Non-linear forward response function g(theta)
def response_function(theta):
    # Example non-linear physical response curve (e.g., natural frequency drop)
    return 10.0 * (1.0 - 0.5 * (theta ** 1.5))

# Initialize uniform prior over bounded domain [0, 1]
current_posterior = np.ones(grid_points) / (theta_max - theta_min)

# Array storage for convergence tracking
running_bayes = []
running_map = []
steps = list(range(n_steps + 1))

# Step 0 estimates
running_bayes.append(np.trapezoid(theta_grid * current_posterior, theta_grid))
running_map.append(theta_grid[np.argmax(current_posterior)])

# True noise-free response
y_true_clean = response_function(theta_true)

# =====================================================================
# SEQUENTIAL BOUNDED GRID UPDATES
# =====================================================================
for k in range(1, n_steps + 1):
    # Simulate noisy sensor measurement y_k
    y_k = y_true_clean + np.random.normal(0, noise_sigma)
    
    # Compute likelihood function across the bounded grid
    model_response = response_function(theta_grid)
    likelihood = stats.norm.pdf(y_k, loc=model_response, scale=noise_sigma)
    
    # Sequential update: Posterior ~ Likelihood * Prior
    current_posterior = current_posterior * likelihood
    
    # Trapezoidal rule normalization
    norm_const = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= norm_const
    
    # Compute point estimators
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior)]
    
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# =====================================================================
# PLOTLY VISUALIZATION
# =====================================================================
fig = go.Figure()

# True damage level reference line
fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Damage (θ = {theta_true})",
    annotation_position="bottom right"
)

# Posterior Mean (Bayes Estimate)
fig.add_trace(go.Scatter(
    x=steps,
    y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# MAP Estimate
fig.add_trace(go.Scatter(
    x=steps,
    y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

fig.update_layout(
    title={
        'text': "SHM: Bayesian Tracking of Structural Damage via Bounded Grid Updates",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sensor Measurement Step (k)",
    yaxis_title="Estimated Damage Parameter (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=5),
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)

fig.show()

```

---

## Interpretation & Convergence Analysis

* **Uncertainty Reduction:** As sensor readings $k$ accumulate, the product of likelihood contributions causes the posterior density curve to narrow significantly around $\theta_{\text{true}}$.
* **Bounded Safety Domain:** Evaluating on a bounded grid strictly prevents out-of-bounds parameter estimates (e.g., negative damage or $>100\%$ stiffness loss).
* **Asymptotic Convergence:** Both the Posterior Mean ($\hat{\theta}_{\text{Bayes}}$) and MAP estimate ($\hat{\theta}_{\text{MAP}}$) converge rapidly toward $\theta_{\text{true}} = 0.65$, demonstrating how sequential Bayesian estimation handles measurement noise to provide reliable real-time structural health monitoring.